# Simulación · Unidad Modular I
## Números aleatorios y pseudoaleatorios, tipos de eventos y simulación de eventos discretos

Este cuaderno acompaña la presentación de la unidad. Cada sección reproduce un ejemplo de las diapositivas con **NumPy**, **pandas**, **Matplotlib** y **seaborn**, y termina con una actividad para practicar.

| Sección | Tema | Diapositivas |
|---|---|---|
| 1 | Números aleatorios U(0, 1) | 4 a 6 |
| 2 | Método de cuadrados medios | 9, 10 y 12 |
| 3 | Método de productos medios | 11 |
| 4 | Congruencial lineal mixto y periodo | 13 a 15 |
| 5 | Congruencial multiplicativo, MINSTD y RANDU | 16 y 17 |
| 6 | Pruebas estadísticas de calidad | 18 y 19 |
| 7 | Transformada inversa | 28 |
| 8 | Simulación de eventos discretos: cola de un cajero | 22, 26 a 29 |
| 9 | Eventos continuos: nivel de un tanque | 23 y 24 |
| 10 | Ejercicios propuestos | 32 y 33 |

**Cómo usarlo:** ejecuta las celdas en orden con `Shift + Enter`, o usa *Entorno de ejecución → Ejecutar todas*.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Paleta de la presentación
MORADO, VERDE, AMBAR, CORAL, AZUL, TINTA = "#6C4AB6", "#00A896", "#F2A900", "#E4572E", "#3A86C8", "#1A1433"
sns.set_theme(style="whitegrid", palette=[MORADO, VERDE, CORAL, AZUL, AMBAR])
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.titleweight": "bold", "axes.titlesize": 13})
pd.set_option("display.precision", 4)

---
## 1. Números aleatorios U(0, 1)

Un número aleatorio $r$ para simulación sigue una distribución uniforme entre 0 y 1:

$$f(r) = 1,\; 0 \le r \le 1 \qquad E(r) = \tfrac{1}{2} \qquad V(r) = \tfrac{1}{12} \approx 0.0833$$

Debe cumplir dos propiedades: **uniformidad** e **independencia**. NumPy trae un generador de alta calidad que usaremos como referencia.

In [ ]:
rng = np.random.default_rng(seed=2026)      # la semilla hace reproducible el experimento
r = rng.random(1000)

print(f"Media    = {r.mean():.4f}   (teórica 0.5000)")
print(f"Varianza = {r.var(ddof=1):.4f}   (teórica {1/12:.4f})")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(r, bins=10, binrange=(0, 1), color=MORADO, ax=ax[0])
ax[0].axhline(len(r) / 10, color=CORAL, ls="--", label="Frecuencia esperada")
ax[0].set(title="Uniformidad: histograma de 1 000 números", xlabel="r", ylabel="Frecuencia"); ax[0].legend()
ax[1].plot(r[:100], marker="o", ms=3, lw=0.8, color=VERDE)
ax[1].set(title="Independencia: primeros 100 valores en orden", xlabel="i", ylabel="r")
plt.tight_layout(); plt.show()

**Reproducibilidad.** Con la misma semilla se obtiene exactamente la misma secuencia. Esa es la gran ventaja de los números *pseudoaleatorios* para depurar y comparar escenarios.

In [ ]:
a = np.random.default_rng(7).random(5)
b = np.random.default_rng(7).random(5)
c = np.random.default_rng(8).random(5)
pd.DataFrame({"semilla 7": a, "semilla 7 (otra vez)": b, "semilla 8": c})

---
## 2. Método de cuadrados medios

Propuesto por John von Neumann (1946):

1. Elegir una semilla $X_0$ de $D$ dígitos.
2. Calcular $Y = X_i^2$ y completar con ceros a la izquierda hasta tener $2D$ dígitos.
3. Tomar los $D$ dígitos centrales: ese es $X_{i+1}$.
4. $r_{i+1} = 0.X_{i+1}$

In [ ]:
def digitos_centrales(y, D):
    """Devuelve (y con ceros a la izquierda, sus D dígitos centrales)."""
    s = str(y).zfill(2 * D)
    if (len(s) - D) % 2:            # si sobra un dígito, se agrega otro cero a la izquierda
        s = "0" + s
    ini = (len(s) - D) // 2
    return s, s[ini:ini + D]

def cuadrados_medios(x0, n, D=4):
    filas, x = [], x0
    for i in range(n):
        y, centro = digitos_centrales(x * x, D)
        filas.append({"i": i, "X_i": str(x).zfill(D), "Y = X_i^2": y, "X_i+1": centro, "r_i+1": float("0." + centro)})
        x = int(centro)
    return pd.DataFrame(filas).set_index("i")

cuadrados_medios(5735, 5)          # ejemplo de la diapositiva 10

### El problema: periodos cortos y degeneración

La siguiente función busca cuántos números se generan antes de que la secuencia empiece a repetirse.

In [ ]:
def periodo(secuencia):
    """Recibe una lista de estados y devuelve (cola, periodo) o None si no se repite."""
    vistos = {}
    for i, estado in enumerate(secuencia):
        if estado in vistos:
            return vistos[estado], i - vistos[estado]
        vistos[estado] = i
    return None

resumen = []
for semilla in [5735, 3708, 3792, 1000, 9999, 1234]:
    df = cuadrados_medios(semilla, 500)
    cola, per = periodo(df["X_i+1"].tolist())
    resumen.append({"semilla": semilla, "valores antes del ciclo": cola, "periodo": per, "ciclo": " → ".join(df["X_i+1"].iloc[cola:cola + min(per, 4)])})
pd.DataFrame(resumen)

In [ ]:
df = cuadrados_medios(5735, 60)
plt.figure(figsize=(10, 3.8))
plt.plot(df.index, df["r_i+1"], marker="o", color=MORADO)
plt.title("Cuadrados medios con semilla 5735: la secuencia cae en un ciclo")
plt.xlabel("i"); plt.ylabel("r"); plt.show()

---
## 3. Método de productos medios

Usa **dos** semillas y multiplica los dos últimos valores:

$$Y_i = X_i \cdot X_{i+1} \qquad X_{i+2} = \text{dígitos centrales de } Y_i \qquad r_{i+1} = 0.X_{i+2}$$

In [ ]:
def productos_medios(x0, x1, n, D=4):
    filas, a, b = [], x0, x1
    for i in range(n):
        y, centro = digitos_centrales(a * b, D)
        filas.append({"i": i, "X_i": str(a).zfill(D), "X_i+1": str(b).zfill(D), "Y_i": y, "X_i+2": centro, "r_i+1": float("0." + centro)})
        a, b = b, int(centro)
    return pd.DataFrame(filas).set_index("i")

productos_medios(5015, 5734, 5)    # ejemplo de la diapositiva 11

In [ ]:
df = productos_medios(5015, 5734, 2000)
estados = list(zip(df["X_i+1"], df["X_i+2"]))     # el estado son los dos últimos valores
print("Valores antes del ciclo y periodo:", periodo(estados))

---
## 4. Método congruencial lineal mixto

$$X_{i+1} = (a\,X_i + c) \bmod m \qquad r_i = \frac{X_i}{m-1}$$

Ejemplo de la diapositiva 14: $a = 19$, $c = 33$, $m = 100$, $X_0 = 37$.

In [ ]:
def congruencial(a, c, m, x0, n):
    # Con c = 0 es el método multiplicativo
    filas, x = [], x0
    for i in range(n):
        t = a * x + c
        nx = t % m
        filas.append({"i": i, "X_i": x, "a·X_i + c": t, "X_i+1": nx, "r_i+1": nx / (m - 1)})
        x = nx
    return pd.DataFrame(filas).set_index("i")

df = congruencial(19, 33, 100, 37, 12)
print("Valores antes del ciclo y periodo:", periodo(df["X_i+1"].tolist()))
df

Solo aparecen 10 valores distintos aunque $m = 100$. El **teorema de Hull y Dobell** dice cuándo se alcanza el periodo completo $N = m$:

1. $c$ y $m$ son primos relativos.
2. $a - 1$ es divisible por todos los factores primos de $m$.
3. Si $m$ es múltiplo de 4, $a - 1$ también lo es.

In [ ]:
from math import gcd

def factores_primos(n):
    f, d = set(), 2
    while d * d <= n:
        while n % d == 0:
            f.add(d); n //= d
        d += 1
    if n > 1: f.add(n)
    return f

def hull_dobell(a, c, m):
    return {"c y m primos relativos": gcd(c, m) == 1,
            "a-1 divisible por factores primos de m": all((a - 1) % p == 0 for p in factores_primos(m)),
            "regla del múltiplo de 4": m % 4 != 0 or (a - 1) % 4 == 0}

casos = {"a=19, c=33, m=100": (19, 33, 100, 37), "a=21, c=33, m=100": (21, 33, 100, 37),
         "a=5, c=7, m=8": (5, 7, 8, 4), "a=5, c=4, m=8": (5, 4, 8, 4)}
tabla = []
for nombre, (a, c, m, x0) in casos.items():
    per = periodo(congruencial(a, c, m, x0, 3 * m)["X_i+1"].tolist())[1]
    tabla.append({"generador": nombre, **hull_dobell(a, c, m), "periodo": per, "m": m})
pd.DataFrame(tabla).set_index("generador")

**Mapa de periodos.** Para $m = 64$ y $c = 7$ se calcula el periodo con cada multiplicador $a$. Solo los valores $a = 1 + 4k$ alcanzan el periodo completo.

In [ ]:
m, c = 64, 7
periodos = pd.DataFrame({"a": range(1, 41)})
periodos["periodo"] = [periodo(congruencial(a, c, m, 1, 3 * m)["X_i+1"].tolist())[1] for a in periodos["a"]]
periodos["completo"] = np.where(periodos["periodo"] == m, "Periodo completo", "Periodo incompleto")

plt.figure(figsize=(12, 4))
sns.barplot(data=periodos, x="a", y="periodo", hue="completo", palette={"Periodo completo": VERDE, "Periodo incompleto": CORAL}, dodge=False)
plt.title("Periodo según el multiplicador a (m = 64, c = 7)"); plt.legend(title=""); plt.show()

---
## 5. Método congruencial multiplicativo

Caso con $c = 0$: $X_{i+1} = (a\,X_i) \bmod m$.

Con $m = 2^g$, $a = 3 + 8k$ o $a = 5 + 8k$ y $X_0$ impar, el periodo máximo es $N = m/4 = 2^{g-2}$.

Ejemplo de la diapositiva 16: $X_0 = 17$, $k = 2$, $g = 5$, es decir $a = 21$, $m = 32$, $N = 8$.

In [ ]:
df = congruencial(21, 0, 32, 17, 10)
print("Periodo:", periodo(df["X_i+1"].tolist())[1])
df

### Generadores reales: MINSTD y RANDU

* **MINSTD** (Park y Miller): $a = 16\,807$, $m = 2^{31} - 1$.
* **RANDU** (IBM, años 60): $a = 65\,539$, $m = 2^{31}$. Es el mal ejemplo clásico: en tres dimensiones sus puntos caen en solo 15 planos.

In [ ]:
def lcg_array(a, c, m, x0, n):
    """Versión rápida que devuelve solo los r como arreglo de NumPy."""
    r, x = np.empty(n), x0
    for i in range(n):
        x = (a * x + c) % m
        r[i] = x / m
    return r

minstd = lcg_array(16807, 0, 2**31 - 1, 12345, 9000)
randu = lcg_array(65539, 0, 2**31, 1, 9000)

fig = plt.figure(figsize=(12, 5.5))
for k, (nombre, r, color) in enumerate([("MINSTD", minstd, VERDE), ("RANDU", randu, CORAL)], start=1):
    ax = fig.add_subplot(1, 2, k, projection="3d")
    ax.scatter(r[0::3], r[1::3], r[2::3], s=2, color=color)
    ax.view_init(elev=28, azim=58)
    ax.set_title(f"{nombre}: ternas (r1, r2, r3)")
plt.tight_layout(); plt.show()

---
## 6. Pruebas estadísticas de calidad

Con un nivel de confianza del 95 % ($\alpha = 0.05$):

| Prueba | Estadístico | Se acepta si |
|---|---|---|
| Medias | $\bar r$ | $\tfrac12 - z_{\alpha/2}\tfrac{1}{\sqrt{12n}} \le \bar r \le \tfrac12 + z_{\alpha/2}\tfrac{1}{\sqrt{12n}}$ |
| Varianza | $S^2$ | $\dfrac{\chi^2_{1-\alpha/2,\,n-1}}{12(n-1)} \le S^2 \le \dfrac{\chi^2_{\alpha/2,\,n-1}}{12(n-1)}$ |
| Uniformidad | $\chi^2_0 = \sum \dfrac{(O_i - E_i)^2}{E_i}$ | $\chi^2_0 \le \chi^2_{\alpha,\,m-1}$ |
| Corridas | $z = \dfrac{C - \mu}{\sigma}$, con $\mu = \tfrac{2n-1}{3}$, $\sigma^2 = \tfrac{16n-29}{90}$ | $\lvert z \rvert \le z_{\alpha/2}$ |

In [ ]:
def pruebas(r, alfa=0.05, clases=10):
    r = np.asarray(r); n = len(r); z = stats.norm.ppf(1 - alfa / 2)
    # Medias
    media = r.mean(); li_m, ls_m = 0.5 - z / np.sqrt(12 * n), 0.5 + z / np.sqrt(12 * n)
    # Varianza
    var = r.var(ddof=1)
    li_v = stats.chi2.ppf(alfa / 2, n - 1) / (12 * (n - 1)); ls_v = stats.chi2.ppf(1 - alfa / 2, n - 1) / (12 * (n - 1))
    # Uniformidad (chi-cuadrada)
    obs, _ = np.histogram(r, bins=clases, range=(0, 1)); esp = n / clases
    chi0 = ((obs - esp) ** 2 / esp).sum(); chi_crit = stats.chi2.ppf(1 - alfa, clases - 1)
    # Corridas arriba y abajo
    signos = np.sign(np.diff(r)); signos = signos[signos != 0]
    corridas = 1 + np.count_nonzero(np.diff(signos))
    mu, sigma = (2 * n - 1) / 3, np.sqrt((16 * n - 29) / 90); z0 = (corridas - mu) / sigma
    return pd.DataFrame([
        {"prueba": "Medias", "estadístico": media, "límite inferior": li_m, "límite superior": ls_m, "pasa": li_m <= media <= ls_m},
        {"prueba": "Varianza", "estadístico": var, "límite inferior": li_v, "límite superior": ls_v, "pasa": li_v <= var <= ls_v},
        {"prueba": "Uniformidad (chi²)", "estadístico": chi0, "límite inferior": 0, "límite superior": chi_crit, "pasa": chi0 <= chi_crit},
        {"prueba": "Independencia (corridas, z)", "estadístico": z0, "límite inferior": -z, "límite superior": z, "pasa": abs(z0) <= z},
    ]).set_index("prueba")

pruebas(minstd[:1000])

### Comparación de generadores

Se generan 1 000 números con cada método y se aplican las cuatro pruebas.

In [ ]:
n = 1000
muestras = {
    "NumPy (PCG64)": np.random.default_rng(1).random(n),
    "MINSTD": minstd[:n],
    "Mixto a=19, c=33, m=100": congruencial(19, 33, 100, 37, n)["r_i+1"].to_numpy(),
    "Cuadrados medios 5735": cuadrados_medios(5735, n)["r_i+1"].to_numpy(),
    "Productos medios": productos_medios(5015, 5734, n)["r_i+1"].to_numpy(),
}
resultado = pd.DataFrame({nombre: pruebas(r)["pasa"] for nombre, r in muestras.items()}).T
resultado["pruebas superadas"] = resultado.sum(axis=1)
resultado

In [ ]:
largo = pd.concat([pd.DataFrame({"r": r, "generador": nombre}) for nombre, r in muestras.items()])
g = sns.displot(data=largo, x="r", col="generador", col_wrap=3, bins=10, binrange=(0, 1), height=2.8, aspect=1.25, color=MORADO)
g.set_titles("{col_name}"); g.figure.suptitle("Uniformidad: histogramas por generador", y=1.03, weight="bold"); plt.show()

fig, ax = plt.subplots(1, len(muestras), figsize=(16, 3.4), sharey=True)
for eje, (nombre, r) in zip(ax, muestras.items()):
    eje.scatter(r[:-1], r[1:], s=5, color=VERDE, alpha=0.6); eje.set_title(nombre, fontsize=10); eje.set(xlabel="r(i)", xlim=(0, 1), ylim=(0, 1))
ax[0].set_ylabel("r(i+1)"); fig.suptitle("Independencia: cada número contra el siguiente", weight="bold", y=1.04); plt.show()

En la gráfica de dispersión, un buen generador llena el cuadrado sin patrones. Los que tienen periodo corto muestran apenas unos pocos puntos repetidos.

---
## 7. Transformada inversa: de $r$ a tiempos aleatorios

Se iguala $r = F(x)$ y se despeja $x$:

| Distribución | Fórmula |
|---|---|
| Exponencial de media $1/\lambda$ | $x = -\tfrac{1}{\lambda}\ln(1 - r)$ |
| Uniforme $(a, b)$ | $x = a + (b - a)\,r$ |

Ejemplo de la diapositiva 28: media de 5 min y $r = 0.35$.

In [ ]:
def exponencial(r, media):
    return -media * np.log(1 - np.asarray(r))

print(f"r = 0.35 → x = {exponencial(0.35, 5):.2f} min")
print(f"r = 0.80 → x = {exponencial(0.80, 5):.2f} min")

x = exponencial(minstd, 5)
plt.figure(figsize=(9, 4))
sns.histplot(x, bins=50, stat="density", color=AZUL)
t = np.linspace(0, x.max(), 300)
plt.plot(t, np.exp(-t / 5) / 5, color=CORAL, lw=2.5, label="Densidad teórica")
plt.title("Tiempos exponenciales generados con MINSTD (media 5 min)"); plt.xlabel("x (min)"); plt.xlim(0, 30); plt.legend(); plt.show()
print(f"Media de la muestra: {x.mean():.3f} min")

---
## 8. Simulación de eventos discretos: cola de un cajero

Un cajero atiende clientes en orden de llegada. Para cada cliente:

$$\text{Inicio} = \max(\text{Llegada},\ \text{Fin anterior}) \qquad \text{Fin} = \text{Inicio} + \text{Servicio} \qquad \text{Espera} = \text{Inicio} - \text{Llegada}$$

Los tiempos entre llegadas (media 4 min) y de servicio (media 3 min) son exponenciales.

In [ ]:
def simular_cola(n, media_llegadas, media_servicio, r):
    """r: arreglo con al menos 2n números U(0,1). Devuelve la tabla de simulación."""
    r = np.clip(np.asarray(r[:2 * n]), 1e-4, 1 - 1e-4)
    r1, r2 = r[0::2], r[1::2]
    entre = exponencial(r1, media_llegadas); servicio = exponencial(r2, media_servicio)
    llegada = entre.cumsum()
    inicio, fin = np.empty(n), np.empty(n)
    fin_anterior = 0.0
    for i in range(n):
        inicio[i] = max(llegada[i], fin_anterior)
        fin[i] = inicio[i] + servicio[i]
        fin_anterior = fin[i]
    return pd.DataFrame({"r1": r1, "entre llegadas": entre, "llegada": llegada, "r2": r2, "servicio": servicio,
                         "inicio": inicio, "fin": fin, "espera": inicio - llegada}, index=pd.RangeIndex(1, n + 1, name="cliente"))

def indicadores(tabla):
    return pd.Series({"espera promedio (min)": tabla["espera"].mean(),
                      "tiempo en el sistema (min)": (tabla["fin"] - tabla["llegada"]).mean(),
                      "utilización del cajero": tabla["servicio"].sum() / tabla["fin"].iloc[-1],
                      "clientes que esperan (%)": 100 * (tabla["espera"] > 0).mean()})

tabla = simular_cola(8, 4, 3, np.random.default_rng(11).random(16))
display(tabla.round(2)); indicadores(tabla).round(3)

### Clientes en el sistema $N(t)$

El estado solo cambia en los instantes de llegada y de salida: por eso la gráfica es **escalonada**.

In [ ]:
def n_en_sistema(tabla):
    eventos = pd.concat([pd.DataFrame({"t": tabla["llegada"], "cambio": 1}), pd.DataFrame({"t": tabla["fin"], "cambio": -1})]).sort_values("t")
    eventos["N"] = eventos["cambio"].cumsum()
    return eventos

tabla = simular_cola(40, 4, 3, np.random.default_rng(11).random(80))
ev = n_en_sistema(tabla)
plt.figure(figsize=(11, 3.8))
plt.step(np.r_[0, ev["t"]], np.r_[0, ev["N"]], where="post", color=CORAL, lw=2)
plt.title("Clientes en el sistema N(t)"); plt.xlabel("Tiempo (min)"); plt.ylabel("Clientes"); plt.show()

### Corridas largas, réplicas y valor teórico

Para una cola M/M/1 con $\lambda = 1/4$ y $\mu = 1/3$:

$$\rho = \frac{\lambda}{\mu} = 0.75 \qquad W_q = \frac{\lambda}{\mu(\mu - \lambda)} = 9 \text{ min}$$

Una sola corrida corta puede alejarse mucho de ese valor. Por eso se hacen **réplicas**.

In [ ]:
Wq_teorico = (1 / 4) / ((1 / 3) * (1 / 3 - 1 / 4))
replicas = []
for n in [50, 500, 5000]:
    for k in range(40):
        t = simular_cola(n, 4, 3, np.random.default_rng(1000 * n + k).random(2 * n))
        replicas.append({"clientes por corrida": n, "réplica": k, "espera promedio": t["espera"].mean()})
replicas = pd.DataFrame(replicas)

plt.figure(figsize=(9, 4.5))
sns.boxplot(data=replicas, x="clientes por corrida", y="espera promedio", color=MORADO, width=0.5)
sns.stripplot(data=replicas, x="clientes por corrida", y="espera promedio", color=TINTA, size=3, alpha=0.5)
plt.axhline(Wq_teorico, color=CORAL, ls="--", label=f"Teórico = {Wq_teorico:.0f} min")
plt.title("40 réplicas por tamaño de corrida"); plt.ylabel("Espera promedio en cola (min)"); plt.legend(); plt.show()
replicas.groupby("clientes por corrida")["espera promedio"].agg(["mean", "std", "min", "max"]).round(2)

### ¿Qué pasa si el generador es malo?

La misma cola alimentada con tres generadores distintos. El mixto de periodo 10 repite siempre los mismos diez números, así que la simulación deja de ser aleatoria.

In [ ]:
n = 2000
fuentes = {"NumPy (PCG64)": np.random.default_rng(5).random(2 * n),
           "MINSTD": lcg_array(16807, 0, 2**31 - 1, 12345, 2 * n),
           "Mixto de periodo 10": congruencial(19, 33, 100, 37, 2 * n)["r_i+1"].to_numpy(),
           "Cuadrados medios 5735": cuadrados_medios(5735, 2 * n)["r_i+1"].to_numpy()}
comparacion = pd.DataFrame({nombre: indicadores(simular_cola(n, 4, 3, r)) for nombre, r in fuentes.items()}).T
comparacion["valores distintos de r"] = [len(np.unique(r)) for r in fuentes.values()]
comparacion.round(3)

---
## 9. Eventos continuos: nivel de un tanque

Aquí el estado cambia en **todo instante** y se describe con una ecuación diferencial:

$$A\,\frac{dh}{dt} = Q_{in} - k\sqrt{h}$$

Se resuelve numéricamente con el **método de Euler**: $h(t + \Delta t) = h(t) + \Delta t\,\dfrac{Q_{in} - k\sqrt{h}}{A}$

El nivel de equilibrio es $h^* = (Q_{in}/k)^2$.

In [ ]:
def tanque(q_in=1.2, k=0.6, A=4.0, h0=0.2, t_max=80, dt=0.05, fallas=None):
    """fallas: lista de intervalos (inicio, fin) en los que la bomba está apagada."""
    t = np.arange(0, t_max + dt, dt); h = np.empty_like(t); h[0] = h0
    for i in range(1, len(t)):
        apagada = fallas is not None and any(a <= t[i - 1] < b for a, b in fallas)
        q = 0.0 if apagada else q_in
        h[i] = max(h[i - 1] + dt * (q - k * np.sqrt(h[i - 1])) / A, 0)
    return pd.DataFrame({"t": t, "h": h})

plt.figure(figsize=(10, 4))
for q in [0.6, 1.2, 1.8]:
    d = tanque(q_in=q)
    plt.plot(d["t"], d["h"], lw=2.2, label=f"Qin = {q}  →  h* = {(q / 0.6) ** 2:.1f} m")
plt.title("Nivel del tanque: cambio suave hacia el equilibrio"); plt.xlabel("Tiempo (min)"); plt.ylabel("Nivel h (m)"); plt.legend(); plt.show()

### Modelo híbrido: eventos discretos sobre un proceso continuo

La bomba falla en instantes aleatorios (tiempo entre fallas exponencial de media 12 min) y la reparación tarda en promedio 4 min. Las fallas son **eventos discretos estocásticos**; el nivel sigue siendo una variable **continua**.

In [ ]:
rng = np.random.default_rng(3)
fallas, t = [], 0.0
while t < 80:
    t += exponencial(rng.random(), 12); dur = exponencial(rng.random(), 4)
    fallas.append((t, t + dur)); t += dur

d = tanque(fallas=fallas)
plt.figure(figsize=(10, 4))
for a, b in fallas:
    plt.axvspan(a, min(b, 80), color=CORAL, alpha=0.18)
plt.plot(d["t"], d["h"], color=VERDE, lw=2.2); plt.axhline(4, color=CORAL, ls="--", label="Equilibrio h* = 4 m")
plt.title("Tanque con fallas aleatorias de la bomba (franjas)"); plt.xlabel("Tiempo (min)"); plt.ylabel("Nivel h (m)"); plt.xlim(0, 80); plt.legend(); plt.show()

---
## 10. Ejercicios propuestos

Resuélvelos primero **a mano** y luego usa las funciones del cuaderno para comprobar.

1. **Cuadrados medios.** Con $X_0 = 3708$ y $D = 4$, genera los primeros cinco $r_i$. ¿Qué ocurre si continúas?
2. **Productos medios.** Con $X_0 = 6965$ y $X_1 = 9803$, genera cinco $r_i$.
3. **Congruencial mixto.** Con $a = 5$, $c = 7$, $m = 8$ y $X_0 = 4$, genera el ciclo completo y verifica Hull y Dobell.
4. **Congruencial multiplicativo.** Con $X_0 = 15$, $k = 1$ ($a = 3 + 8k$) y $g = 6$, calcula $a$, $m$ y el periodo esperado.
5. **Prueba de medias.** Una muestra de $n = 40$ números tiene promedio 0.43. Con $\alpha = 5\,\%$, ¿se acepta que la media es 0.5?
6. **Transformada inversa.** Media de 5 min entre llegadas. Calcula el tiempo para $r = 0.35$ y $r = 0.80$.
7. **Clasificación.** Indica si es discreto o continuo: inventario de una tienda, temperatura de un horno, llamadas en espera, nivel de un embalse.
8. **Cola manual.** Entre llegadas: 2, 4, 1, 3, 5 min. Servicios: 3, 2, 4, 1, 2 min. Calcula la espera promedio y la utilización.

**Retos con código**

* Cambia la media de servicio de la cola a 3.8 min. ¿Qué pasa con la espera? ¿Y con 4.2 min?
* Busca todas las semillas de 4 dígitos cuyo método de cuadrados medios tenga más de 80 valores distintos.
* Agrega un segundo cajero a `simular_cola` y compara la espera promedio.

In [ ]:
# Comprobación de los ejercicios 1 a 4
print("1)", cuadrados_medios(3708, 5)["r_i+1"].tolist())
print("   ciclo:", cuadrados_medios(3708, 12)["X_i+1"].tolist())
print("2)", productos_medios(6965, 9803, 5)["r_i+1"].tolist())
print("3)", congruencial(5, 7, 8, 4, 8)["X_i+1"].tolist(), hull_dobell(5, 7, 8))
e4 = congruencial(11, 0, 64, 15, 40)["X_i+1"].tolist()
print("4) a = 11, m = 64, periodo =", periodo(e4)[1])

In [ ]:
# Comprobación de los ejercicios 5, 6 y 8
n, z = 40, stats.norm.ppf(0.975)
li, ls = 0.5 - z / np.sqrt(12 * n), 0.5 + z / np.sqrt(12 * n)
print(f"5) Límites: [{li:.4f}, {ls:.4f}] → 0.43 {'se acepta' if li <= 0.43 <= ls else 'se rechaza'}")
print(f"6) {exponencial(0.35, 5):.2f} min y {exponencial(0.80, 5):.2f} min")

entre, servicio = np.array([2, 4, 1, 3, 5]), np.array([3, 2, 4, 1, 2])
llegada = entre.cumsum(); inicio = np.empty(5); fin = np.empty(5); previo = 0
for i in range(5):
    inicio[i] = max(llegada[i], previo); fin[i] = inicio[i] + servicio[i]; previo = fin[i]
manual = pd.DataFrame({"llegada": llegada, "servicio": servicio, "inicio": inicio, "fin": fin, "espera": inicio - llegada})
display(manual)
print(f"8) Espera promedio = {manual['espera'].mean():.2f} min, utilización = {servicio.sum() / fin[-1]:.1%}")

---
### Para seguir aprendiendo

* Abre el archivo `Laboratorio_Simulacion_Unidad1.html` para ver estos mismos modelos de forma animada.
* Biblioteca recomendada para modelos más grandes de eventos discretos en Python: **SimPy**.